In [1]:
# set up main path where everything will be you should download the
# hugging face directory described in readme and put it here on the same
# server where the data analyzer is run so that the data analyzer code with 
# the GPU can access these files
# You should replace the below path with your location
import json
import os
try:
    with open('../conf/config.json', 'r') as f:
        config = json.load(f)
    storage_path = config['storage_path']
    data_path = config['data_path']
    derivatives_path = config['derivatives_path']
    fsl_path = config['fsl_path']
    project_path = config['project_path']
    assert os.path.exists(storage_path), "The specified storage path does not exist."
    assert os.path.exists(data_path), "The specified data path does not exist."
    assert os.path.exists(derivatives_path), "The specified derivatives path does not exist."
    assert os.path.exists(fsl_path), "The specified FSL path does not exist."
    assert os.path.exists(project_path), "The specified project path does not exist."
except FileNotFoundError:
    raise FileNotFoundError("config.json file not found. Please create it with the required paths.")

"""-----------------------------------------------------------------------------
Imports and set up for mindEye
-----------------------------------------------------------------------------"""

import sys
import shutil
import argparse
import numpy as np
import math
import time
import random
import string
import h5py
from scipy import stats
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms
from accelerate import Accelerator, DeepSpeedPlugin
# SDXL unCLIP requires code from https://github.com/Stability-AI/generative-models/tree/main
sys.path.append(f'{project_path}/models/generative_models')
sys.path.append(f'{project_path}/models')
import sgm
from generative_models.sgm.modules.encoders.modules import FrozenOpenCLIPImageEmbedder, FrozenOpenCLIPEmbedder2
from generative_models.sgm.models.diffusion import DiffusionEngine
from generative_models.sgm.util import append_dims
from omegaconf import OmegaConf
from PIL import Image
# tf32 data type is faster than standard float32
torch.backends.cuda.matmul.allow_tf32 = True
# custom functions #
import utils_mindeye
from models import *
import pandas as pd
import ants
import nilearn
import nibabel as nib
import pdb
from nilearn.plotting import plot_design_matrix
### Multi-GPU config ###
local_rank = os.getenv('RANK')
if local_rank is None: 
    local_rank = 0
else:
    local_rank = int(local_rank)
accelerator = Accelerator(split_batches=False, mixed_precision="fp16")
device = accelerator.device

%load_ext autoreload
%autoreload 2

/teamspace/studios/this_studio/rtcloud-projects/mindeye/conf/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


line 6:  /teamspace/studios/this_studio/rtcloud-projects/mindeye/scripts
line 6:  /teamspace/studios/this_studio/rtcloud-projects/mindeye/scripts
line 14:  /teamspace/studios/this_studio/rtcloud-projects/mindeye/scripts
line 14:  /teamspace/studios/this_studio/rtcloud-projects/mindeye/scripts


In [2]:
if accelerator.mixed_precision == "bf16":
    data_type = torch.bfloat16
elif accelerator.mixed_precision == "fp16":
    data_type = torch.float16
else:
    data_type = torch.float32

In [3]:
sub = "sub-005"
session = "ses-03"
task = 'C'  # 'study' or 'A'; used to search for functional run in bids format
func_task_name = 'C'  # 'study' or 'A'; used to search for functional run in bids format
n_runs = 11
realtime = True # load realtime betas or offline betas

ses_list = [session]
design_ses_list = [session]
    
task_name = f"_task-{task}" if task != 'study' else ''
designdir = f"{data_path}/events"

In [4]:
# Unlike the real-time saved beta, the glmsingle results are not masked so we need to load mask
union_mask = np.load(f"{data_path}/ses-01_MST_split_rels.npy")#union_mask_from_ses-01-02.npy")
union_mask = union_mask>0.2 # only for the ses 1 mask which was unthresholed
mask_img = nib.load(f'{data_path}/{sub}_final_mask.nii.gz')  # nsdgeneral mask in functional space
assert mask_img.get_fdata().sum() == union_mask.shape

In [5]:
if realtime:
    # Load pre-calculated betas (real-time processing)
    # I made a mistake that I start run-02 after I finished run-01 so I am doing a dirty fix here:
    derivatives_path = '/teamspace/studios/this_studio/rtcloud-projects/mindeye/derivatives'
    all_betas = []
    for run_num in (1,n_runs): # np.arange(1,n_runs+1)we could just load the last run, this is for sanity check only
        betas_fname = f'{sub}_{session}_task-{task}_run-{run_num:02d}_recons/betas_run-{run_num:02d}.npy'
        print(betas_fname)
        betas = np.load(os.path.join(derivatives_path,betas_fname))
        print(betas.shape)  
        all_betas.append(betas)
    all_betas = np.concatenate(all_betas,axis=0)
    print(all_betas.shape)
    # hf_derivatives_path = '/teamspace/studios/this_studio/rtcloud-projects/mindeye/derivatives' #'/teamspace/studios/this_studio/rtcloud-projects/mindeye/derivatives' # 
    # # all_betas = []
    # for run_num in np.arange(1,n_runs+1): # np.arange(1,n_runs+1)we could just load the last run, this is for sanity check only
    #     betas_fname = f'{sub}_{session}_task-{task}_run-{run_num:02d}_recons/betas_run-{run_num:02d}.npy'
    #     print(betas_fname)
    #     betas = np.load(os.path.join(hf_derivatives_path,betas_fname))
    #     print(betas.shape)  
    # #     all_betas.append(betas)
    # all_betas = betas # It looks like the all_betas is aggregating across all runs so we'll just take run 5 output which includes all previous runs
    z_mean = np.mean(np.array(all_betas), axis=0)
    z_std = np.std(np.array(all_betas), axis=0)
    zscored_all_betas = ((np.array(all_betas) - z_mean) / (z_std + 1e-6))
else:
    # Alternatively: Load precalculated betas (fmriprep + glmsingle)
    orig_glmsingle_path ='/teamspace/gcs_folders/share/real_time_mindeye_data/glmsingle'
    glmsingle_output = f'{orig_glmsingle_path}/glmsingle_{sub}_{session}{task_name}/TYPED_FITHRF_GLMDENOISE_RR.npz'
    vox = np.squeeze(np.load(glmsingle_output)['betasmd']).T

    # Unmask from brain voxels to whole image and apply the final_mask to the image to get the final masked voxels:
    if vox.shape[1] != int(mask_img.get_fdata().sum()):
        # betas probably come from a session-specific mask that doesn't correspond to the intersection mask created during multi-session analysis
        # if so, try reshaping vox based on the session-specific mask
        print('vox doesn\'t match roi shape; reshaping to match multi-session final mask')
        session_mask = nib.load(f'{orig_glmsingle_path}/glmsingle_{sub}_{session}{task_name}/{sub}_{session}{task_name}_brain.nii.gz')
        assert int(session_mask.get_fdata().sum()) == vox.shape[1], 'session mask doesn\'t correspond to glmsingle betas!'
        unmasked_vox = nilearn.masking.unmask(vox, session_mask)  # shape will be (X x Y x Z x images)
        vox = nilearn.masking.apply_mask(unmasked_vox, mask_img)
        print(vox.shape) 

    # all_betas2 = vox[:,union_mask==1]
    all_betas2 = vox[:,union_mask>.2]
    print(all_betas2.shape)
    z_mean = np.mean(np.array(all_betas2), axis=0)
    z_std = np.std(np.array(all_betas2), axis=0)
    zscored_all_betas = ((np.array(all_betas2) - z_mean) / (z_std + 1e-6))

sub-005_ses-03_task-C_run-01_recons/betas_run-01.npy
(63, 2792)
sub-005_ses-03_task-C_run-11_recons/betas_run-11.npy
(630, 2792)
(693, 2792)


In [6]:
# nilearn.plotting.plot_img(mask_img)

In [7]:
# Check how similar the two are
# plt.scatter(all_betas.flatten(),all_betas2.flatten())

In [85]:
data, starts, _, is_new_run, image_names, unique_images, len_unique_images = utils_mindeye.load_design_files(
    sub=sub,
    session=session,
    func_task_name=task,
    designdir=designdir,
    design_ses_list=design_ses_list
)

non_blank = []
for im in image_names:
    if 'blank' not in im:
        non_blank.append(im)

print(len(non_blank))
print(non_blank[:5])
assert(len(non_blank)==zscored_all_betas.shape[0])

if sub == 'sub-001':
    if session == 'ses-01':
        assert image_names[0] == 'images/image_686_seed_1.png'
    elif session in ('ses-02', 'all'):
        assert image_names[0] == 'all_stimuli/special515/special_40840.jpg'
    elif session == 'ses-03':
        assert image_names[0] == 'all_stimuli/special515/special_69839.jpg'
    elif session == 'ses-04':
        assert image_names[0] == 'all_stimuli/rtmindeye_stimuli/image_686_seed_1.png'
elif sub == 'sub-003':
    assert image_names[0] == 'all_stimuli/rtmindeye_stimuli/image_686_seed_1.png'

unique_images = np.unique(image_names.astype(str))
unique_images = unique_images[(unique_images!="nan")]
len_unique_images = len(unique_images)

if (sub == 'sub-001' and session == 'ses-04') or (sub == 'sub-003' and session == 'ses-01'):
    assert len(unique_images) == 851

image_idx = np.array([])  # contains the unique index of each presented image
vox_image_names = np.array([])  # contains the names of the images corresponding to image_idx
all_special515_images = dict()
for i, im in enumerate(image_names):
    if "all_stimuli/special515" not in str(im):
        i+=1
        continue

    vox_image_names = np.append(vox_image_names, im)
            
    image_idx_ = np.where(im==unique_images)[0].item()
    image_idx = np.append(image_idx, image_idx_)
    
    all_special515_images[i] = im
    i+=1
    
image_idx = torch.Tensor(image_idx).long()

unique_special515_images = np.unique(list(all_special515_images.values())) 
print(len(unique_special515_images))

Data shape: (780, 126)
Using design file: /teamspace/gcs_folders/share/real_time_mindeye_data/3t_data/data/events/csv/sub-005_ses-03.csv
Total number of images: 770
Number of unique images: 532
693
['all_stimuli/unchosen_nsd_1000_images/unchosen_7211_cocoid_59250.png', 'all_stimuli/special515/special_67295.jpg', 'all_stimuli/unchosen_nsd_1000_images/unchosen_5729_cocoid_53029.png', 'all_stimuli/special515/special_70232.jpg', 'all_stimuli/unchosen_nsd_1000_images/unchosen_7251_cocoid_26645.png']
50


In [86]:
# Load ground truth images
import imageio.v2 as imageio
resize_transform = transforms.Resize((224, 224))
images = None
for im_name in tqdm(unique_special515_images):
    im = imageio.imread(f"{data_path}/{im_name}")
    im = torch.Tensor(im / 255).permute(2,0,1)
    im = resize_transform(im.unsqueeze(0))
    if images is None:
        images = im
    else:
        images = torch.vstack((images, im))

  0%|          | 0/50 [00:00<?, ?it/s]

/teamspace/studios/this_studio/rtcloud-projects/mindeye/conf/.venv/lib/python3.11/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
100%|██████████| 50/50 [00:02<00:00, 19.60it/s]


In [10]:
# I think the previous way I did it is problematic, it did not take into account of the blank images?
# design_data = pd.read_csv(f'{data_path}/events/csv/{sub}_{session}.csv')
# max_Trials = zscored_all_betas.shape[0]
# run_Trials = np.max(design_data.trial_index) #63 for ses-6 but 69 for ses-1

# all_beta_indices,all_run_indices,all_trial_indices = [],[],[]
# for curr_img in unique_special515_images:
#     cond = design_data.current_image == curr_img
#     beta_idx = np.array((design_data.run_num[cond])*run_Trials+design_data.trial_index[cond]) # the loaded beta are from run-02 to run-05 but in the csv run_num == 1 is run-02
#     beta_idx = beta_idx[beta_idx<max_Trials].astype(int) 
#     # either
#     all_beta_indices.append(beta_idx) # this is the overall index in all runs, it's easier if we already saved the betas
#     # or
#     all_run_indices.append(np.array(design_data.run_num[cond].astype(int))) # this is which run (zero-indexing), best if the betas are saved per run, or we are calculating it with per run events
#     all_trial_indices.append(np.array(design_data.trial_index[cond].astype(int))) # this is which trial in that run, best if the betas are saved per run, or we are calculating it

In [73]:
ndscore_events = [pd.read_csv(f'{data_path}/events/{sub}_{session}_task-{func_task_name}_run-{run+1:02d}_events.tsv', sep = "\t", header = 0) for run in range(n_runs)] 
ndscore_tr_labels = [pd.read_csv(f"{data_path}/events/{sub}_{session}_task-{func_task_name}_run-{run+1:02d}_tr_labels.csv") for run in range(n_runs)]

In [74]:
run_num = 1
events_df = ndscore_events[run_num - 1]
tr_labels_hrf = ndscore_tr_labels[run_num - 1]["tr_label_hrf"].tolist()
events_df = events_df[events_df['image_name'] != 'blank.jpg']  # must drop blank.jpg after tr_labels_hrf is defined to keep indexing consistent

In [91]:
len(tr_labels_hrf)

192

770

In [64]:
all_beta_indices = []
for curr_img in unique_special515_images:
    beta_idx = np.where(curr_img == np.array(non_blank))[0]
    beta_idx = beta_idx.astype(int) 
    all_beta_indices.append(beta_idx)
all_beta_indices = np.array(all_beta_indices)
print(all_beta_indices)

[[375 599 666]
 [ 83 222 601]
 [124 392 656]
 [ 37 120 486]
 [ 66 270 297]
 [469 555 650]
 [215 479 642]
 [  8 443 658]
 [193 288 588]
 [378 406 428]
 [ 87 184 453]
 [ 36 326 511]
 [267 377 591]
 [122 418 690]
 [ 59 264 403]
 [190 648 659]
 [421 558 683]
 [196 539 640]
 [ 82 148 439]
 [ 79 219 286]
 [188 436 476]
 [200 616 664]
 [312 426 636]
 [ 93 363 530]
 [  5  22 151]
 [ 71 407 543]
 [307 448 457]
 [ 88 228 673]
 [291 315 662]
 [163 277 424]
 [425 501 631]
 [147 400 495]
 [191 309 521]
 [134 202 466]
 [280 672 675]
 [109 241 572]
 [118 340 347]
 [153 247 386]
 [515 525 623]
 [156 266 685]
 [537 583 585]
 [519 532 629]
 [346 551 556]
 [  1  69 293]
 [172 508 603]
 [  3  30 427]
 [181 194 582]
 [ 53 231 647]
 [ 49 353 686]
 [198 633 646]]


Prepare the mindeye model for reconstruction

In [11]:
cache_dir= f"{storage_path}/cache"
model_name="sub-005_ses-01_task-C_bs24_MST_rishab_repeats_3split_3_avgrepeats_finalmask_epochs_45" #"sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_3_avgrepeats_finalmask_epochs_45" # "sub-005_ses-01_task-C_bs24_MST_rishab_MSTsplit_0_avgrepeats_finalmask" #"sub-005_ses-01-03_task-C_bs24_MST_rishab_MSTsplit_unionmask_ses-01-03_finetune"
subj=1
hidden_dim=1024
blurry_recon = False
n_blocks=4 
seq_len = 1
total_beta_voxels = np.sum(union_mask)
if realtime:
    save_path_root = f"{derivatives_path}/{sub}_{session}_task-{func_task_name}_recons_byrepeats_rt/{model_name}"
else:
    save_path_root = f"{derivatives_path}/{sub}_{session}_task-{func_task_name}_recons_byrepeats/{model_name}"
os.makedirs(save_path_root,exist_ok=True)
print(save_path_root)

/teamspace/studios/this_studio/rtcloud-projects/mindeye/derivatives/sub-005_ses-03_task-C_recons_byrepeats_rt/sub-005_ses-01_task-C_bs24_MST_rishab_repeats_3split_3_avgrepeats_finalmask_epochs_45


In [12]:
import pickle
with open(f"{storage_path}/clip_img_embedder", "rb") as input_file:
    clip_img_embedder = pickle.load(input_file)
clip_img_embedder.to(device)
clip_seq_dim = 256
clip_emb_dim = 1664

In [13]:
class MindEyeModule(nn.Module):
    def __init__(self):
        super(MindEyeModule, self).__init__()
    def forward(self, x):
        return x

model = MindEyeModule()

class RidgeRegression(torch.nn.Module):
    # make sure to add weight_decay when initializing optimizer
    def __init__(self, input_sizes, out_features, seq_len): 
        super(RidgeRegression, self).__init__()
        self.out_features = out_features
        self.linears = torch.nn.ModuleList([
                torch.nn.Linear(input_size, out_features) for input_size in input_sizes
            ])
    def forward(self, x, subj_idx):
        out = torch.cat([self.linears[subj_idx](x[:,seq]).unsqueeze(1) for seq in range(seq_len)], dim=1)
        return out
num_voxels = total_beta_voxels
model.ridge = RidgeRegression([num_voxels], out_features=hidden_dim, seq_len=seq_len)

from diffusers.models.vae import Decoder
class BrainNetwork(nn.Module):
    def __init__(self, h=4096, in_dim=15724, out_dim=768, seq_len=2, n_blocks=n_blocks, drop=.15, 
                clip_size=768):
        super().__init__()
        self.seq_len = seq_len
        self.h = h
        self.clip_size = clip_size

        self.mixer_blocks1 = nn.ModuleList([
            self.mixer_block1(h, drop) for _ in range(n_blocks)
        ])
        self.mixer_blocks2 = nn.ModuleList([
            self.mixer_block2(seq_len, drop) for _ in range(n_blocks)
        ])

        # Output linear layer
        self.backbone_linear = nn.Linear(h * seq_len, out_dim, bias=True) 
        self.clip_proj = self.projector(clip_size, clip_size, h=clip_size)


    def projector(self, in_dim, out_dim, h=2048):
        return nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.GELU(),
            nn.Linear(in_dim, h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Linear(h, h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Linear(h, out_dim)
        )

    def mlp(self, in_dim, out_dim, drop):
        return nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(out_dim, out_dim),
        )

    def mixer_block1(self, h, drop):
        return nn.Sequential(
            nn.LayerNorm(h),
            self.mlp(h, h, drop),  # Token mixing
        )

    def mixer_block2(self, seq_len, drop):
        return nn.Sequential(
            nn.LayerNorm(seq_len),
            self.mlp(seq_len, seq_len, drop)  # Channel mixing
        )

    def forward(self, x):
        # make empty tensors
        c,b,t = torch.Tensor([0.]), torch.Tensor([[0.],[0.]]), torch.Tensor([0.])

        # Mixer blocks
        residual1 = x
        residual2 = x.permute(0,2,1)
        for block1, block2 in zip(self.mixer_blocks1,self.mixer_blocks2):
            x = block1(x) + residual1
            residual1 = x
            x = x.permute(0,2,1)

            x = block2(x) + residual2
            residual2 = x
            x = x.permute(0,2,1)

        x = x.reshape(x.size(0), -1)
        backbone = self.backbone_linear(x).reshape(len(x), -1, self.clip_size)
        c = self.clip_proj(backbone)

        return backbone, c, b

model.backbone = BrainNetwork(h=hidden_dim, in_dim=hidden_dim, seq_len=seq_len, 
                        clip_size=clip_emb_dim, out_dim=clip_emb_dim*clip_seq_dim) 
utils_mindeye.count_params(model.ridge)
utils_mindeye.count_params(model.backbone)
utils_mindeye.count_params(model)

# setup diffusion prior network
out_dim = clip_emb_dim
depth = 6
dim_head = 52
heads = clip_emb_dim//52 # heads * dim_head = clip_emb_dim
timesteps = 100

prior_network = PriorNetwork(
        dim=out_dim,
        depth=depth,
        dim_head=dim_head,
        heads=heads,
        causal=False,
        num_tokens = clip_seq_dim,
        learned_query_mode="pos_emb"
    )

model.diffusion_prior = BrainDiffusionPrior(
    net=prior_network,
    image_embed_dim=out_dim,
    condition_on_text_encodings=False,
    timesteps=timesteps,
    cond_drop_prob=0.2,
    image_embed_scale=None,
)
model.to(device)

utils_mindeye.count_params(model.diffusion_prior)
utils_mindeye.count_params(model)

param counts:
2,860,032 total
2,860,032 trainable
param counts:
453,360,280 total
453,360,280 trainable
param counts:
456,220,312 total
456,220,312 trainable
param counts:
259,865,216 total
259,865,200 trainable
param counts:
716,085,528 total
716,085,512 trainable


716085512

In [14]:
outdir = f'{data_path}/model'
checkpoint = torch.load(outdir+f'/{model_name}.pth', map_location='cpu')
state_dict = checkpoint['model_state_dict']
model.load_state_dict(state_dict, strict=True)
del checkpoint

In [15]:
# prep unCLIP
config = OmegaConf.load(f"{project_path}/models/generative_models/configs/unclip6.yaml")
config = OmegaConf.to_container(config, resolve=True)
unclip_params = config["model"]["params"]
network_config = unclip_params["network_config"]
denoiser_config = unclip_params["denoiser_config"]
# first_stage_config = unclip_params["first_stage_config"]
conditioner_config = unclip_params["conditioner_config"]
sampler_config = unclip_params["sampler_config"]
scale_factor = unclip_params["scale_factor"]
disable_first_stage_autocast = unclip_params["disable_first_stage_autocast"]
offset_noise_level = unclip_params["loss_fn_config"]["params"]["offset_noise_level"]
# first_stage_config['target'] = 'sgm.models.autoencoder.AutoencoderKL'
sampler_config['params']['num_steps'] = 38
with open(f"{storage_path}/diffusion_engine", "rb") as input_file:
    diffusion_engine = pickle.load(input_file)
# set to inference
diffusion_engine.eval().requires_grad_(False)
diffusion_engine.to(device)
ckpt_path = f'{cache_dir}/unclip6_epoch0_step110000.ckpt'
ckpt = torch.load(ckpt_path, map_location='cpu')
diffusion_engine.load_state_dict(ckpt['state_dict'])
batch={"jpg": torch.randn(1,3,1,1).to(device), # jpg doesnt get used, it's just a placeholder
    "original_size_as_tuple": torch.ones(1, 2).to(device) * 768,
    "crop_coords_top_left": torch.zeros(1, 2).to(device)}
out = diffusion_engine.conditioner(batch)
vector_suffix = out["vector"].to(device)
# f = h5py.File(f'{storage_path}/coco_images_224_float16.hdf5', 'r')
# images = f['images']

In [16]:
def do_reconstructions(betas_tt):
    """
    takes in the beta map for a stimulus trial in torch tensor format (tt)

    returns reconstructions and clipvoxels for retrievals
    """
    # start_reconstruction_time = time.time()
    model.to(device)
    model.eval().requires_grad_(False)
    clipvoxelsTR = None
    reconsTR = None
    num_samples_per_image = 1
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16):
        voxel = betas_tt
        voxel = voxel.to(device)
        voxel_ridge = model.ridge(voxel[:,[0]],0) # 0th index of subj_list
        backbone0, clip_voxels0, blurry_image_enc0 = model.backbone(voxel_ridge)
        clip_voxels = clip_voxels0
        backbone = backbone0
        blurry_image_enc = blurry_image_enc0[0]
        clipvoxelsTR = clip_voxels.cpu()
        prior_out = model.diffusion_prior.p_sample_loop(backbone.shape, 
                        text_cond = dict(text_embed = backbone), 
                        cond_scale = 1., timesteps = 20)  
        for i in range(len(voxel)):
            samples = utils_mindeye.unclip_recon(prior_out[[i]],
                            diffusion_engine,
                            vector_suffix,
                            num_samples=num_samples_per_image)
            if reconsTR is None:
                reconsTR = samples.cpu()
            else:
                reconsTR = torch.vstack((reconsTR, samples.cpu()))
            imsize = 224
            reconsTR = transforms.Resize((imsize,imsize), antialias=True)(reconsTR).float().numpy().tolist()
        return reconsTR, clipvoxelsTR
    
def batchwise_cosine_similarity(Z,B):
    Z = Z.flatten(1)
    B = B.flatten(1).T
    Z_norm = torch.linalg.norm(Z, dim=1, keepdim=True)  # Size (n, 1).
    B_norm = torch.linalg.norm(B, dim=0, keepdim=True)  # Size (1, b).
    cosine_similarity = ((Z @ B) / (Z_norm @ B_norm)).T
    return cosine_similarity

def get_top_retrievals(clipvoxel, all_images, total_retrievals = 1):
    '''
    clipvoxel: output from do_recons that contains that information needed for retrievals
    all_images: all ground truth actually seen images by the participant in day 2 run 1

    outputs the top retrievals
    '''
    values_dict = {}
    with torch.cuda.amp.autocast(dtype=torch.float16):
        emb = clip_img_embedder(torch.reshape(all_images,(all_images.shape[0], 3, 224, 224)).to(device)).float() # CLIP-Image
        emb = emb.cpu()
        emb_ = clipvoxel # CLIP-Brain
        emb = emb.reshape(len(emb),-1)
        emb_ = np.reshape(emb_, (1, 425984))
        emb = nn.functional.normalize(emb,dim=-1)
        emb_ = nn.functional.normalize(emb_,dim=-1)
        emb_ = emb_.float()
        fwd_sim = batchwise_cosine_similarity(emb_,emb)  # brain, clip
        print("Given Brain embedding, find correct Image embedding")
    fwd_sim = np.array(fwd_sim.cpu())
    which = np.flip(np.argsort(fwd_sim, axis = 0))
    imsize = 224
    for attempt in range(total_retrievals):
        values_dict[f"attempt{(attempt+1)}"] = transforms.Resize((imsize,imsize), antialias=True)(all_images[which[attempt].copy()]).float().numpy().tolist()
    return values_dict

def convert_image_array_to_PIL(image_array):
    if image_array.ndim == 4:
        image_array = image_array[0]

    # get the dimension to h, w, 3|1
    if image_array.ndim == 3 and image_array.shape[0] == 3:
        image_array = np.transpose(image_array, (1, 2, 0))  # Change shape to (height, width, 3)
    
    # clip the image array to 0-1
    image_array = np.clip(image_array, 0, 1)
    # convert the image array to uint8
    image_array = (image_array * 255).astype('uint8')
    # convert the image array to PIL
    return Image.fromarray(image_array)


In [65]:
# Seed everything
utils_mindeye.seed_everything(0)

In [18]:
# # get the mask and the reference files
# ndscore_events = [pd.read_csv(f'{data_path}/events/{sub}_{session}_task-{func_task_name}_run-{run+1:02d}_events.tsv', sep = "\t", header = 0) for run in range(n_runs)]  # create a new list of events_df's which will have the trial_type modified to be unique identifiers
# ndscore_tr_labels = [pd.read_csv(f"{data_path}/events/{sub}_{session}_task-{func_task_name}_run-{run+1:02d}_tr_labels.csv") for run in range(n_runs)]
# tr_length = 1.5

# run_num = 2
# events_df = ndscore_events[run_num - 1]

In [ ]:
# image_counter = 0 # 0-49
# irep = 0 # 0-2
# stimulus_trial_counter = all_trial_indices[image_counter][irep]

# print(events_df[events_df.trial_number==stimulus_trial_counter])
# print(events_df[events_df.trial_number==stimulus_trial_counter].onset//tr_length)


In [20]:
# n_trs = 192
# for TR in range(n_trs-1):
#     events_df['onset'] = events_df['onset'].astype(float)

#     run_start_time = events_df['onset'].iloc[0]
#     events_df['onset'] -= run_start_time

#     cropped_events = events_df[events_df.onset <= TR*tr_length]
#     cropped_events = cropped_events.copy()
#     cropped_events.loc[:, 'trial_type'] = np.where(cropped_events['trial_number'] == stimulus_trial_counter, "probe", "reference")
#     cropped_events = cropped_events.drop(columns=['is_correct', 'image_name', 'response_time', 'trial_number'])

    

Run through all special 515 images (50, 3 repeats)

In [68]:
# The key loop that loops through all MST_images and save the recons from 1 to max num of repeats
for image_id,image_name in enumerate(unique_special515_images):
    image_name = os.path.basename(image_name).split('.')[0] # strip the folders and suffix
    save_path = f"{save_path_root}/{image_name}"
    os.makedirs(save_path,exist_ok=True)
    print(f"Calculating reconstructions for {image_name}")
    
    beta_idx = all_beta_indices[image_id]
     
    all_recons_save = []
    all_clipvoxels_save = []
    all_ground_truth_save = []
    all_retrieved_save = []
    for irep,id in enumerate(beta_idx):
        betas = np.mean(zscored_all_betas[beta_idx[:irep+1]],axis=0) # average the betas for 1,2,3,4.. repeats
        betas = betas[np.newaxis, np.newaxis, :]
        betas_tt = torch.Tensor(betas).to("cpu")
        
        reconsTR, clipvoxelsTR = do_reconstructions(betas_tt) # got an error here
        if clipvoxelsTR is None:
            with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16):
                voxel = betas_tt
                voxel = voxel.to(device)
                assert voxel.shape[1] == 1
                voxel_ridge = model.ridge(voxel[:,[-1]],0) # 0th index of subj_list
                backbone0, clip_voxels0, blurry_image_enc0 = model.backbone(voxel_ridge)
                clip_voxels = clip_voxels0
                backbone = backbone0
                blurry_image_enc = blurry_image_enc0[0]
                clipvoxelsTR = clip_voxels.cpu()
        values_dict = get_top_retrievals(clipvoxelsTR, all_images=images, total_retrievals=5)
        image_array = np.array(reconsTR)[0]
        # If the image has 3 channels (RGB), you need to reorder the dimensions
        if image_array.ndim == 3 and image_array.shape[0] == 3:
            image_array = np.transpose(image_array, (1, 2, 0))  # Change shape to (height, width, 3)
        all_recons_save.append(image_array)
        all_clipvoxels_save.append(clipvoxelsTR)
        all_ground_truth_save.append(images[image_id].numpy())
        all_retrieved_save.append([np.array(value) for key, value in values_dict.items() if (not ('ground_truth' in key))])

    all_recons_save_tensor = torch.tensor(all_recons_save).permute(0,3,1,2)
    all_clipvoxels_save_tensor = torch.stack(all_clipvoxels_save, dim=0)
    all_ground_truth_save_tensor = torch.tensor(all_ground_truth_save)
    all_retrieved_save_tensor = torch.stack([torch.tensor(np.array(item)) for item in all_retrieved_save], dim=0)
    torch.save(all_recons_save_tensor, os.path.join(save_path, "all_recons.pt"))
    torch.save(all_clipvoxels_save_tensor, os.path.join(save_path, "all_clipvoxels.pt"))
    torch.save(all_ground_truth_save_tensor, os.path.join(save_path, "all_ground_truth.pt"))
    torch.save(all_retrieved_save_tensor, os.path.join(save_path, "all_retrieved.pt"))
    print("all_recons_save_tensor.shape: ", all_recons_save_tensor.shape)
    print("all_clipvoxels_save_tensor.shape: ", all_clipvoxels_save_tensor.shape)
    print("all_ground_truth_save_tensor.shape: ", all_ground_truth_save_tensor.shape)
    print("all_retrieved_save_tensor.shape: ", all_retrieved_save_tensor.shape)
    print("All tensors saved successfully on ", save_path)
            

Calculating reconstructions for special_11942


sampling loop time step:   0%|          | 0/19 [00:00<?, ?it/s]

/teamspace/studios/this_studio/rtcloud-projects/mindeye/conf/.venv/lib/python3.11/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/teamspace/studios/this_studio/rtcloud-projects/mindeye/conf/.venv/lib/python3.11/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Given Brain embedding, find correct Image embedding


sampling loop time step:   0%|          | 0/19 [00:00<?, ?it/s]

...

Given Brain embedding, find correct Image embedding


sampling loop time step:   0%|          | 0/19 [00:00<?, ?it/s]

Given Brain embedding, find correct Image embedding


sampling loop time step:   0%|          | 0/19 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# def plot_discrete_hist():

# plt.hist(data, bins=np.arange(data.min() - 0.5, data.max() + 1.5, 1))
# plt.xticks(range(data.min(), data.max() + 1))
# plt.xlabel("Value")
# plt.ylabel("Count")
# plt.title("Discrete Histogram")
# plt.show()

In [ ]:
# plt.hist(MST_ID, bins=np.arange(MST_ID.min() - 0.5, MST_ID.max() + 1.5, 1))